# AIE-CASE: color-invariant saree design recognition

The task is face recognition for textiles: find the saree whose **design** matches a query photo, regardless of the **colorway** it was woven in.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Open the DeepLure Drive folder, click **▾ next to "sarees_dataset" → Organise → Add shortcut → My Drive**. The notebook reads it straight from your Drive; nothing is re-uploaded anywhere.
3. *(optional)* Add a Colab secret 🔑 `KAGGLE_API_TOKEN` (your Kaggle token) to also use `div456/indian-saree-patterns`. Public datasets usually download without it.

Then *Runtime → Run all*. Checkpoints and results are saved to `MyDrive/saree_runs`, so a disconnect loses nothing.

## Approach note (≤500 characters)
ConvNeXt-Atto (ImageNet) → GeM → 256-d L2 embedding. There are no design labels, so each deduplicated image is its own design. Training uses SupCon on 4 views per image, each with a random crop, a synthetic recolor (k-means Lab palette remap/permute/transfer, HSV, channel, gray) and capture noise. A palette bank shared across the batch puts different motifs in identical colors, so color cannot separate classes. Inference: resize, center crop, flip TTA, cosine, per-design max.

In [ ]:
import os, sys, subprocess, torch
print(torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True); import timm
print('timm', timm.__version__)
os.makedirs('sareeid', exist_ok=True)
NW = os.cpu_count()  # dataloader workers

from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/saree_runs'   # runs + results persist on Drive
WORK = '/content/work'                    # image cache on fast local disk
os.makedirs(W, exist_ok=True)

## Package source
The same code as the `sareeid/` package in the repo, written to disk here so the notebook is self-contained. Cells are collapsed.

In [ ]:
%%writefile sareeid/__init__.py
"""Color-invariant saree design recognition."""

In [ ]:
%%writefile sareeid/recolor.py
"""Synthetic colorway generation.

A saree "design" is the spatial layout of its motifs; a "colorway" is the palette the
loom fills that layout with. We simulate new colorways by segmenting the image into its
dominant color regions (k-means in CIELAB) and re-painting each region, while keeping the
per-pixel residual (weave texture, shading, folds) so the result still looks like fabric.

Generators are split into two families:
  TRAIN_FAMILIES  - used as training augmentation
  HELDOUT_FAMILIES - used ONLY in evaluation, to check that invariance generalises beyond
                     the exact augmentations the model was trained on.

All functions take / return uint8 RGB arrays (H, W, 3) and an np.random.Generator.
"""
from __future__ import annotations

import cv2
import numpy as np

Palette = np.ndarray  # (k, 3) float32 Lab (OpenCV 8-bit scale: L,a,b in 0..255)


def to_lab(img: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)


def from_lab(lab: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)


def kmeans_lab(lab: np.ndarray, k: int, rng: np.random.Generator, iters: int = 8,
               sample: int = 4096) -> tuple[Palette, np.ndarray]:
    """Tiny k-means on a pixel subsample, then full-image assignment. Returns
    centroids sorted by lightness and an (H, W) label map."""
    px = lab.reshape(-1, 3)
    s = px[rng.choice(len(px), min(sample, len(px)), replace=False)]
    c = s[rng.choice(len(s), k, replace=False)].copy()
    for _ in range(iters):
        a = ((s[:, None, :] - c[None]) ** 2).sum(-1).argmin(1)
        for j in range(k):
            m = a == j
            if m.any():
                c[j] = s[m].mean(0)
    order = np.argsort(c[:, 0])
    c = c[order]
    labels = ((px[:, None, :] - c[None]) ** 2).sum(-1).argmin(1)
    return c, labels.reshape(lab.shape[:2])


def extract_palette(img: np.ndarray, k: int = 6, seed: int = 0) -> Palette:
    return kmeans_lab(to_lab(img), k, np.random.default_rng(seed))[0]


def random_palette(k: int, rng: np.random.Generator, min_dist: float = 35.0) -> Palette:
    """Random Lab palette whose colors stay mutually distinguishable, so a recolor never
    merges two motif regions into one (which would genuinely change the design)."""
    cols: list[np.ndarray] = []
    for _ in range(200 * k):
        c = np.array([rng.uniform(25, 235), 128 + rng.normal(0, 40), 128 + rng.normal(0, 40)],
                     np.float32)
        if all(np.linalg.norm(c - o) >= min_dist for o in cols):
            cols.append(c)
            if len(cols) == k:
                break
    while len(cols) < k:  # fall back to spreading lightness
        cols.append(np.array([255 * len(cols) / k, 128, 128], np.float32))
    return np.stack(cols)


def _fit_palette(src: Palette, tgt: Palette) -> Palette:
    """Resample a lightness-sorted target palette to len(src) entries (rank matching)."""
    tgt = tgt[np.argsort(tgt[:, 0])]
    idx = np.round(np.linspace(0, len(tgt) - 1, len(src))).astype(int)
    return tgt[idx]


def palette_remap(img: np.ndarray, rng: np.random.Generator, target: Palette | None = None,
                  k: int | None = None, mode: str = "random", shuffle_prob: float = 0.3) -> np.ndarray:
    """Repaint each k-means color region.
    mode='random'  : fresh random palette
    mode='permute' : shuffle the image's own colors between regions (gold-on-red -> red-on-gold)
    mode='transfer': map regions to `target` by lightness rank (palette of another saree)
    """
    lab = to_lab(img)
    k = k or int(rng.integers(3, 8))
    cent, _ = kmeans_lab(lab, k, rng)
    cent = merge_close(cent)  # shading of one dye must not become two colours
    k = len(cent)
    if mode == "transfer" and target is not None:
        new = _fit_palette(cent, target)
        if rng.random() < shuffle_prob:  # also allow non-rank-preserving assignments
            new = new[rng.permutation(k)]
    elif mode == "permute":
        new = cent[rng.permutation(k)]
    else:
        new = random_palette(k, rng)
    # soft assignment: pixels between two dyes blend smoothly instead of snapping (no blotches)
    w = soft_assign(lab, cent)
    resid = lab - w @ cent
    resid[..., 1:] *= rng.uniform(0.3, 1.0)  # keep texture in L, damp old chroma
    return from_lab(w @ new + resid)


def _lab_dist2(x: np.ndarray, c: np.ndarray) -> np.ndarray:
    """Squared Lab distance with lightness down-weighted: folds/shadows change L, not the dye."""
    d = x[..., None, :] - c
    return 0.35 * d[..., 0] ** 2 + d[..., 1] ** 2 + d[..., 2] ** 2


def merge_close(cent: Palette, thr: float = 22.0) -> Palette:
    cent = [c.copy() for c in cent]
    merged = True
    while merged and len(cent) > 2:
        merged = False
        C = np.stack(cent)
        D = np.sqrt(_lab_dist2(C, C)) + np.eye(len(C)) * 1e9
        i, j = np.unravel_index(D.argmin(), D.shape)
        if D[i, j] < thr:
            cent[i] = (C[i] + C[j]) / 2
            cent.pop(j)
            merged = True
    C = np.stack(cent)
    return C[np.argsort(C[:, 0])]


def soft_assign(lab: np.ndarray, cent: Palette, tau: float = 60.0) -> np.ndarray:
    d = _lab_dist2(lab, cent)
    d -= d.min(-1, keepdims=True)
    w = np.exp(-d / tau)
    return w / w.sum(-1, keepdims=True)


def hsv_shift(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + rng.uniform(0, 180)) % 180
    hsv[..., 1] = np.clip(hsv[..., 1] * rng.uniform(0.4, 1.6), 0, 255)
    hsv[..., 2] = 255 * (hsv[..., 2] / 255) ** rng.uniform(0.6, 1.6)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def channel_shuffle_invert(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = img[..., rng.permutation(3)].copy()
    inv = rng.random(3) < 0.35
    out[..., inv] = 255 - out[..., inv]
    return out


def gray(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if rng.random() < 0.3:
        g = 255 - g
    return np.repeat(g[..., None], 3, -1)


# ---- held-out (evaluation-only) generators -------------------------------------------

def channel_curves(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Random monotone tone curve per RGB channel (direction random) - a 'dye lot' change."""
    out = np.empty_like(img)
    xs = np.linspace(0, 255, 6)
    for ch in range(3):
        ys = np.sort(rng.uniform(0, 255, 6))
        ys[0], ys[-1] = rng.uniform(0, 60), rng.uniform(195, 255)
        if rng.random() < 0.5:
            ys = ys[::-1]
        lut = np.interp(np.arange(256), xs, ys).astype(np.uint8)
        out[..., ch] = lut[img[..., ch]]
    return out


def reinhard_transfer(img: np.ndarray, rng: np.random.Generator,
                      ref: np.ndarray | None = None) -> np.ndarray:
    """Global Lab mean/std transfer (Reinhard et al. 2001) to a reference or random target."""
    lab = to_lab(img)
    mu, sd = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0) + 1e-3
    if ref is not None:
        r = to_lab(ref).reshape(-1, 3)
        tmu, tsd = r.mean(0), r.std(0) + 1e-3
    else:
        tmu = np.array([rng.uniform(70, 190), 128 + rng.normal(0, 30), 128 + rng.normal(0, 30)])
        tsd = sd * rng.uniform(0.6, 1.5, 3)
    return from_lab((lab - mu) / sd * tsd + tmu)


TRAIN_FAMILIES = ("palette_random", "palette_permute", "palette_transfer", "hsv", "channel", "gray")
HELDOUT_FAMILIES = ("curves", "reinhard")


def apply_family(img: np.ndarray, family: str, rng: np.random.Generator,
                 target: Palette | None = None, ref: np.ndarray | None = None) -> np.ndarray:
    if family == "identity":
        return img
    if family == "palette_random":
        return palette_remap(img, rng, mode="random")
    if family == "palette_permute":
        return palette_remap(img, rng, mode="permute")
    if family == "palette_transfer":
        return palette_remap(img, rng, target=target, mode="transfer" if target is not None else "random")
    if family == "hsv":
        return hsv_shift(img, rng)
    if family == "channel":
        return channel_shuffle_invert(img, rng)
    if family == "gray":
        return gray(img, rng)
    if family == "curves":
        return channel_curves(img, rng)
    if family == "reinhard":
        return reinhard_transfer(img, rng, ref)
    raise ValueError(family)


TRAIN_WEIGHTS = {"identity": 0.08, "palette_random": 0.25, "palette_permute": 0.12,
                 "palette_transfer": 0.30, "hsv": 0.12, "channel": 0.08, "gray": 0.05}


def random_train_recolor(img: np.ndarray, rng: np.random.Generator,
                         target: Palette | None = None) -> np.ndarray:
    fams, w = zip(*TRAIN_WEIGHTS.items())
    fam = fams[rng.choice(len(fams), p=np.array(w) / sum(w))]
    return apply_family(img, fam, rng, target=target)

In [ ]:
%%writefile sareeid/data.py
"""Data preparation and datasets.

prepare(): scan raw roots -> cache resized copies -> group near-duplicates -> split by group.
The split is done on *groups*, so two photos of the same physical saree can never end up on
both sides of the train/test boundary (the most common leakage bug in retrieval benchmarks).
"""
from __future__ import annotations

import csv
import hashlib
import os
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from . import recolor as rc

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


def read_rgb(path: str | os.PathLike) -> np.ndarray | None:
    buf = np.fromfile(str(path), np.uint8)  # np.fromfile handles non-ASCII Windows paths
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR) if buf.size else None
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def resize_short(img: np.ndarray, short: int) -> np.ndarray:
    h, w = img.shape[:2]
    s = short / min(h, w)
    if s >= 1:
        return img
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)


def phash(img: np.ndarray) -> list[int]:
    """64-bit DCT perceptual hashes on grayscale for all 8 rotations/flips (index 0 = as-is).
    Training treats rotated/flipped copies as the same design, so a rotated copy of a test image
    sitting in train would leak - grouping therefore matches against every orientation."""
    g0 = cv2.resize(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (32, 32), interpolation=cv2.INTER_AREA)
    hs = []
    for g in (g0, g0[:, ::-1]):
        for r in range(4):
            d = cv2.dct(np.ascontiguousarray(np.rot90(g, r)).astype(np.float32))[:8, :8].flatten()
            hs.append(int("".join("1" if b else "0" for b in d > np.median(d[1:])), 2))
    return hs


_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def _popcount(x: np.ndarray) -> np.ndarray:
    return _POP8[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)


def source_stem(path: str) -> str:
    """Roboflow exports save several augmented copies of one photo as `<name>_jpg.rf.<hash>.jpg`;
    everything before `.rf.` identifies the source photo."""
    name = Path(path).name
    return name.split(".rf.")[0] if ".rf." in name else name


def _group_by_hash(hashes: list[list[int]], thr: int, keys: list[str] | None = None) -> list[int]:
    """Union-find over pairs whose min Hamming distance (over the 8 orientations) <= thr,
    plus pairs sharing the same source key (e.g. Roboflow augmentations of one photo)."""
    n = len(hashes)
    H = np.array(hashes, dtype=np.uint64)  # (n, 8)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(n - 1):
        dist = _popcount(np.bitwise_xor(H[i + 1:, :1], H[i][None, :])).min(1)
        for j in np.nonzero(dist <= thr)[0]:
            a, b = find(i), find(i + 1 + int(j))
            if a != b:
                parent[b] = a
    if keys is not None:
        first: dict[str, int] = {}
        for i, k in enumerate(keys):
            if k in first:
                a, b = find(first[k]), find(i)
                if a != b:
                    parent[b] = a
            else:
                first[k] = i
    roots = [find(i) for i in range(n)]
    remap = {r: k for k, r in enumerate(dict.fromkeys(roots))}
    return [remap[r] for r in roots]


def prepare(roots: list[str], out_dir: str, short: int = 320, min_side: int = 96,
            hash_thr: int = 6, split=(0.7, 0.1, 0.2), seed: int = 0) -> str:
    """Build cache + manifest.csv (columns: path, source, orig, group, split)."""
    out = Path(out_dir)
    (out / "cache").mkdir(parents=True, exist_ok=True)
    rows, seen_md5 = [], set()
    n_bad = n_small = n_dup = 0
    for root in roots:
        root_p = Path(root)
        # kagglehub caches datasets as .../<slug>/versions/<n>: name the source by the slug
        src_name = root_p.parent.parent.name if root_p.name.isdigit() and root_p.parent.name == "versions" else root_p.name
        files = sorted(p for p in root_p.rglob("*") if p.suffix.lower() in IMG_EXT and p.is_file())
        print(f"[prepare] {root} ({src_name}): {len(files)} image files")
        for n, p in enumerate(files, 1):
            if n % 500 == 0:
                print(f"  {n}/{len(files)}", flush=True)
            raw = p.read_bytes()  # read once: network drives (Colab/Drive) are slow
            md5 = hashlib.md5(raw).hexdigest()
            if md5 in seen_md5:  # byte-identical file listed twice
                n_dup += 1
                continue
            bgr = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR) if raw else None
            img = None if bgr is None else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            if img is None:
                n_bad += 1
                continue
            if min(img.shape[:2]) < min_side:
                n_small += 1
                continue
            seen_md5.add(md5)
            img = resize_short(img, short)
            cp = out / "cache" / f"{md5}.jpg"
            if not cp.exists():
                cv2.imwrite(str(cp), cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
            rows.append({"path": str(cp), "source": src_name, "orig": str(p), "hash": phash(img)})
    keys = [r["source"] + "/" + source_stem(r["orig"]) if ".rf." in r["orig"] else r["orig"] for r in rows]
    if len(rows) < 10:
        raise RuntimeError(f"only {len(rows)} usable images found under {roots} "
                           f"({n_bad} unreadable, {n_small} too small) - check the dataset paths")
    groups = _group_by_hash([r["hash"] for r in rows], hash_thr, keys)
    rng = np.random.default_rng(seed)
    ug = rng.permutation(max(groups) + 1)
    n_tr, n_va = int(split[0] * len(ug)), int(split[1] * len(ug))
    split_of = {g: ("train" if i < n_tr else "val" if i < n_tr + n_va else "test") for i, g in enumerate(ug)}
    for r, g in zip(rows, groups):
        r["group"], r["split"] = g, split_of[g]
        del r["hash"]
    mpath = out / "manifest.csv"
    with open(mpath, "w", newline="", encoding="utf8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "source", "orig", "group", "split"])
        w.writeheader()
        w.writerows(rows)
    multi = sum(1 for g in set(groups) if groups.count(g) > 1)
    print(f"[prepare] kept {len(rows)} images in {len(set(groups))} groups "
          f"({multi} groups with >1 photo) | dropped: {n_bad} unreadable, {n_small} too small, "
          f"{n_dup} byte-duplicates")
    for s in ("train", "val", "test"):
        print(f"  {s}: {sum(r['split'] == s for r in rows)} images")
    return str(mpath)


@dataclass
class Item:
    path: str
    group: int
    split: str
    source: str


def load_manifest(path: str, split: str | None = None) -> list[Item]:
    with open(path, encoding="utf8") as f:
        items = [Item(r["path"], int(r["group"]), r["split"], r["source"]) for r in csv.DictReader(f)]
    return [i for i in items if split is None or i.split == split]


# ---- geometric / photometric augmentation (numpy, applied to uint8 RGB) --------------

def random_resized_crop(img, size, rng, scale=(0.35, 1.0), ratio=(0.75, 1.333)):
    h, w = img.shape[:2]
    for _ in range(10):
        area = h * w * rng.uniform(*scale)
        ar = np.exp(rng.uniform(np.log(ratio[0]), np.log(ratio[1])))
        cw, ch = int(round(np.sqrt(area * ar))), int(round(np.sqrt(area / ar)))
        if 0 < cw <= w and 0 < ch <= h:
            x, y = rng.integers(0, w - cw + 1), rng.integers(0, h - ch + 1)
            return cv2.resize(img[y:y + ch, x:x + cw], (size, size), interpolation=cv2.INTER_AREA)
    return center_crop(img, size)


def center_crop(img, size):
    img = resize_short(img, size) if min(img.shape[:2]) > size else cv2.resize(
        img, None, fx=size / min(img.shape[:2]), fy=size / min(img.shape[:2]))
    h, w = img.shape[:2]
    y, x = (h - size) // 2, (w - size) // 2
    return np.ascontiguousarray(img[y:y + size, x:x + size])


def geo_photo_aug(img, rng, strength=1.0):
    """Orientation-free fabric augmentation + capture artefacts (blur, noise, JPEG, light)."""
    if rng.random() < 0.5:
        img = img[:, ::-1]
    img = np.rot90(img, int(rng.integers(0, 4)))
    if rng.random() < 0.5 * strength:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-20, 20), rng.uniform(1.0, 1.25))
        img = cv2.warpAffine(np.ascontiguousarray(img), M, (w, h), borderMode=cv2.BORDER_REFLECT)
    img = np.ascontiguousarray(img)
    if rng.random() < 0.3 * strength:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.3, 1.5))
    if rng.random() < 0.3 * strength:  # uneven illumination (shadow / fold)
        h, w = img.shape[:2]
        gx = np.linspace(rng.uniform(0.6, 1.0), rng.uniform(1.0, 1.3), w)[None, :, None]
        img = np.clip(img * gx, 0, 255).astype(np.uint8)
    if rng.random() < 0.2 * strength:
        img = np.clip(img + rng.normal(0, 6, img.shape), 0, 255).astype(np.uint8)
    if rng.random() < 0.3 * strength:
        _, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(40, 90))])
        img = cv2.imdecode(enc, cv2.IMREAD_UNCHANGED)
    return img


def to_tensor(img: np.ndarray, gray: bool = False) -> torch.Tensor:
    if gray:
        img = np.repeat(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[..., None], 3, -1)
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy())


class ImageCache:
    """Per-worker LRU-free memo of decoded cached images (the cache is already small)."""

    def __init__(self):
        self.mem: dict[str, np.ndarray] = {}

    def __call__(self, path):
        if path not in self.mem:
            self.mem[path] = read_rgb(path)
        return self.mem[path]


class MultiViewTrainSet(Dataset):
    """Returns K independently recolored + augmented views of one image (one design).

    Palette sharing: recolor targets are drawn from a small *palette bank* that is shared by
    the whole batch, so the batch is full of different designs wearing identical palettes.
    In the contrastive loss those are negatives - colour alone can no longer separate them.
    """

    def __init__(self, items: list[Item], size=224, views=4, bank_size=24, seed=0, gray=False):
        self.items, self.size, self.views, self.gray = items, size, views, gray
        self.load = ImageCache()
        rng = np.random.default_rng(seed)
        print(f"[train] extracting palettes for {len(items)} images ...")
        self.palettes = [rc.extract_palette(read_rgb(it.path), k=6, seed=i) for i, it in enumerate(items)]
        self.bank_size = bank_size
        self.set_epoch(0)
        self._rng = rng
        # one label per group: multiple photos of the same saree are positives too
        uniq = {g: i for i, g in enumerate(dict.fromkeys(it.group for it in items))}
        self.labels = [uniq[it.group] for it in items]

    def set_epoch(self, epoch: int):
        if self.bank_size == 0:  # ablation: independent per-sample targets, no sharing
            self.bank = None
            return
        rng = np.random.default_rng(10_000 + epoch)
        n_real = self.bank_size // 2
        real = [self.palettes[i] for i in rng.choice(len(self.palettes), n_real)]
        self.bank = real + [rc.random_palette(6, rng) for _ in range(self.bank_size - n_real)]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rng = np.random.default_rng()
        img = self.load(self.items[idx].path)
        views = []
        for _ in range(self.views):
            v = random_resized_crop(img, self.size, rng)
            if self.bank is not None:
                target = self.bank[rng.integers(len(self.bank))]
            elif rng.random() < 0.5:
                target = self.palettes[rng.integers(len(self.palettes))]
            else:
                target = rc.random_palette(6, rng)
            v = rc.random_train_recolor(v, rng, target=target)
            v = geo_photo_aug(v, rng)
            views.append(to_tensor(v, self.gray))
        return torch.stack(views), self.labels[idx]

In [ ]:
%%writefile sareeid/model.py
"""Embedding network: pretrained timm backbone -> GeM pooling -> linear projection -> L2 norm."""
from __future__ import annotations

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F


class GeM(nn.Module):
    """Generalised-mean pooling (Radenovic et al. 2018). p=1 is avg, p->inf is max; learnable p
    lets the net emphasise sparse, salient motif activations over plain background."""

    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p))
        self.eps = eps

    def forward(self, x):
        # fp32 so x**p cannot overflow under fp16 autocast
        return x.float().clamp(min=self.eps).pow(self.p).mean((-2, -1)).pow(1.0 / self.p)


class SareeEmbedder(nn.Module):
    def __init__(self, backbone: str = "convnext_atto.d2_in1k", dim: int = 256, pretrained: bool = True):
        super().__init__()
        self.backbone_name, self.dim = backbone, dim
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="")
        c = self.backbone.num_features
        self.pool = GeM()
        self.head = nn.Sequential(nn.Linear(c, dim), nn.BatchNorm1d(dim)) if dim else nn.Identity()

    def features(self, x):
        f = self.backbone(x)
        if f.ndim == 3:  # ViT tokens (B, N, C) -> drop prefix tokens, average patches
            f = f[:, getattr(self.backbone, "num_prefix_tokens", 1):].mean(1)
            return f
        return self.pool(f)

    def forward(self, x):
        return F.normalize(self.head(self.features(x)), dim=-1)


def load_checkpoint(path: str, map_location="cpu") -> tuple[SareeEmbedder, dict]:
    ck = torch.load(path, map_location=map_location, weights_only=False)
    m = SareeEmbedder(ck["backbone"], ck["dim"], pretrained=False)
    m.load_state_dict(ck["state_dict"])
    return m.eval(), ck


def zero_shot_model(backbone: str) -> SareeEmbedder:
    """Pretrained backbone + GeM, no projection - the 'no training' baseline."""
    return SareeEmbedder(backbone, dim=0, pretrained=True).eval()

In [ ]:
%%writefile sareeid/losses.py
import torch
import torch.nn.functional as F


def supcon_loss(z: torch.Tensor, labels: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """Multi-positive supervised contrastive loss (Khosla et al. 2020).

    z: (N, D) L2-normalised embeddings, labels: (N,) design ids. Every other view of the same
    design is a positive; every view of any other design (including the ones rendered in the
    *same* palette, thanks to the shared palette bank) is a negative.
    """
    z = F.normalize(z.float(), dim=-1)
    sim = z @ z.T / temperature
    n = z.shape[0]
    self_mask = torch.eye(n, dtype=torch.bool, device=z.device)
    pos = (labels[:, None] == labels[None, :]) & ~self_mask
    sim = sim.masked_fill(self_mask, float("-inf"))
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    n_pos = pos.sum(1)
    loss = -(log_prob.masked_fill(~pos, 0).sum(1)) / n_pos.clamp(min=1)
    return loss[n_pos > 0].mean()

In [ ]:
%%writefile sareeid/benchmark.py
"""Deterministic colour-invariance benchmark built from a held-out split.

Every benchmark image is described by a small spec and rendered on the fly with a fixed seed,
so the benchmark is exactly reproducible, never stored on disk, and identical for every model.

Protocols (gallery always = one clean, centre-cropped image per held-out design):
  P1 colorway  - queries = each design re-coloured by EVERY generator family (6 train + 2
                 held-out), random crop/rotation/blur/JPEG. Measures R@1/R@5/mAP per family.
  P2 trap      - query = design d painted in the palette of another design d' (the 'donor').
                 A colour-biased model retrieves d'. We report R@1 and Trap@1 (top-1 == donor).
  P3 shared    - gallery = all designs in palette A, queries = all designs in palette B. Colour
                 carries zero information; only the motif can identify. Averaged over R draws.
  P4 rephoto   - real second photos of the same saree (near-duplicate groups), if any.
Verification pairs:
  genuine            (clean d, P1 query of d)            - cross-palette, same motif
  impostor_random    (clean d, P1 query of d')           - different motif
  impostor_samepal   (d in palette A, d' in palette A)   - different motif, IDENTICAL palette
"""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import Dataset

from . import recolor as rc
from .data import Item, ImageCache, center_crop, geo_photo_aug, random_resized_crop, read_rgb, to_tensor

FAMILIES = rc.TRAIN_FAMILIES + rc.HELDOUT_FAMILIES


def build_specs(items: list[Item], seed: int = 0, shared_rounds: int = 3,
                pairs_per_design: int = 2, size: int = 224) -> dict:
    rng = np.random.default_rng(seed)
    # one representative per group = the design; remaining group members = real re-photos
    reps, extra = {}, []
    for it in items:
        (extra.append(it) if it.group in reps else reps.__setitem__(it.group, it))
    designs = list(reps.values())
    n = len(designs)
    assert n >= 3, "need at least 3 held-out designs"
    paths = [d.path for d in designs]
    S = []  # all specs; roles reference indices into S

    def add(**kw):
        S.append(kw)
        return len(S) - 1

    def other(i):
        j = int(rng.integers(n - 1))
        return j + (j >= i)

    gallery = [add(op="orig", path=paths[i], label=i) for i in range(n)]
    p1 = {f: [] for f in FAMILIES}
    for i in range(n):
        for f in FAMILIES:
            d = other(i)
            p1[f].append(add(op="colorway", path=paths[i], label=i, family=f, donor=paths[d],
                             donor_label=d, seed=int(rng.integers(2**31))))
    trap = []
    for i in range(n):
        d = other(i)
        trap.append(add(op="colorway", path=paths[i], label=i, family="palette_transfer",
                        donor=paths[d], donor_label=d, seed=int(rng.integers(2**31)), strict=True))
    shared = []
    for r in range(shared_rounds):
        pa, pb = int(rng.integers(2**31)), int(rng.integers(2**31))
        g = [add(op="palette", path=paths[i], label=i, pal_seed=pa, seed=int(rng.integers(2**31)), aug=False)
             for i in range(n)]
        q = [add(op="palette", path=paths[i], label=i, pal_seed=pb, seed=int(rng.integers(2**31)), aug=True)
             for i in range(n)]
        shared.append((g, q))
    rephoto = [add(op="orig", path=it.path, label=[d.group for d in designs].index(it.group))
               for it in extra]

    # verification pairs (indices into S)
    gen, imp_r, imp_s = [], [], []
    allq = [(f, k) for f in FAMILIES for k in range(n)]
    for i in range(n):
        for _ in range(pairs_per_design):
            f = FAMILIES[int(rng.integers(len(FAMILIES)))]
            gen.append((gallery[i], p1[f][i]))
            j = other(i)
            imp_r.append((gallery[i], p1[f][j]))
            g, _ = shared[int(rng.integers(shared_rounds))]
            imp_s.append((g[i], g[other(i)]))
    return dict(specs=S, n_designs=n, gallery=gallery, p1=p1, trap=trap, shared=shared,
                rephoto=rephoto, pairs=dict(genuine=gen, impostor_random=imp_r, impostor_samepal=imp_s),
                size=size, n_queries_p1=len(allq))


class SpecDataset(Dataset):
    def __init__(self, specs: list[dict], size: int = 224, gray: bool = False):
        self.specs, self.size, self.gray = specs, size, gray
        self.load = ImageCache()
        self.pal_cache: dict = {}

    def __len__(self):
        return len(self.specs)

    def render(self, s: dict) -> np.ndarray:
        img = self.load(s["path"])
        if s["op"] == "orig":
            return center_crop(img, self.size)
        rng = np.random.default_rng(s["seed"])
        if s["op"] == "palette":
            v = center_crop(img, self.size)
            pal = rc.random_palette(6, np.random.default_rng(s["pal_seed"]))
            v = rc.palette_remap(v, rng, target=pal, k=6, mode="transfer", shuffle_prob=0.0)
            return geo_photo_aug(random_resized_crop(v, self.size, rng, scale=(0.6, 1.0)), rng, 0.7) \
                if s["aug"] else v
        # op == colorway
        v = random_resized_crop(img, self.size, rng, scale=(0.5, 1.0))
        donor = self.load(s["donor"])
        if s["donor"] not in self.pal_cache:
            self.pal_cache[s["donor"]] = rc.extract_palette(donor, k=6, seed=0)
        if s.get("strict"):
            v = rc.palette_remap(v, rng, target=self.pal_cache[s["donor"]], k=6, mode="transfer", shuffle_prob=0.0)
        else:
            v = rc.apply_family(v, s["family"], rng, target=self.pal_cache[s["donor"]],
                                ref=center_crop(donor, self.size))
        return geo_photo_aug(v, rng, 0.7)

    def __getitem__(self, i):
        return to_tensor(self.render(self.specs[i]), self.gray), i


@torch.no_grad()
def embed_specs(model, specs, device, batch_size=128, workers=2, gray=False, tta=False, size=224):
    ds = SpecDataset(specs, size=size, gray=gray)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, num_workers=workers,
                                     persistent_workers=False)
    out = None
    model.eval()
    for x, idx in dl:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            z = model(x).float()
            if tta:
                z = torch.nn.functional.normalize(z + model(x.flip(-1)).float(), dim=-1)
        if out is None:
            out = torch.empty(len(ds), z.shape[1])
        out[idx] = z.cpu()
    return out

In [ ]:
%%writefile sareeid/metrics.py
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, roc_curve


def identification(q: torch.Tensor, q_lab, g: torch.Tensor, g_lab, ks=(1, 5, 10), donor_lab=None) -> dict:
    """Rank the gallery for every query. Returns Recall@k (CMC), mAP and optionally Trap@1."""
    q_lab, g_lab = torch.as_tensor(q_lab), torch.as_tensor(g_lab)
    sim = q @ g.T
    order = sim.argsort(dim=1, descending=True)
    hits = (g_lab[order] == q_lab[:, None])  # (Q, G) bool in rank order
    out = {f"R@{k}": hits[:, :k].any(1).float().mean().item() for k in ks if k <= g.shape[0]}
    cum = hits.float().cumsum(1)
    ranks = torch.arange(1, hits.shape[1] + 1).float()
    ap = ((cum / ranks) * hits).sum(1) / hits.sum(1).clamp(min=1)
    out["mAP"] = ap.mean().item()
    if donor_lab is not None:
        out["Trap@1"] = (g_lab[order[:, 0]] == torch.as_tensor(donor_lab)).float().mean().item()
    out["n_queries"] = int(q.shape[0])
    return out


def verification(scores: np.ndarray, same: np.ndarray, threshold: float | None = None) -> dict:
    scores, same = np.asarray(scores, float), np.asarray(same, bool)
    fpr, tpr, thr = roc_curve(same, scores)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    out = {"AUC": float(roc_auc_score(same, scores)), "EER": float((fpr[i] + fnr[i]) / 2),
           "thr_at_EER": float(thr[i])}
    for far in (1e-2, 1e-3):
        ok = fpr <= far
        out[f"TAR@FAR={far:g}"] = float(tpr[ok].max()) if ok.any() else 0.0
    if threshold is not None:  # threshold calibrated on the validation split
        pred = scores >= threshold
        out["threshold"] = float(threshold)
        out["acc@thr"] = float((pred == same).mean())
        out["TAR@thr"] = float(pred[same].mean())
        out["FAR@thr"] = float(pred[~same].mean())
    out["n_genuine"], out["n_impostor"] = int(same.sum()), int((~same).sum())
    return out

In [ ]:
%%writefile sareeid/evaluate.py
"""Run the colour-invariance benchmark for a trained checkpoint or a zero-shot baseline.

  python -m sareeid.evaluate --manifest work/manifest.csv --ckpt runs/atto/best.pt
  python -m sareeid.evaluate --manifest work/manifest.csv --zero-shot convnext_atto.d2_in1k [--gray]

The verification threshold is calibrated on the *val* split (EER point) and then applied
unchanged to *test*, so the reported accuracy is not tuned on test data.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch

from .benchmark import FAMILIES, build_specs, embed_specs
from .data import load_manifest
from .metrics import identification, verification
from .model import load_checkpoint, zero_shot_model
from . import recolor as rc


def run_benchmark(model, items, device, gray=False, seed=0, threshold=None, workers=2,
                  batch_size=128, tta=False, size=224) -> dict:
    B = build_specs(items, seed=seed, size=size)
    S = B["specs"]
    Z = embed_specs(model, S, device, batch_size=batch_size, workers=workers, gray=gray, tta=tta, size=size)
    lab = lambda idx: [S[i]["label"] for i in idx]
    g = B["gallery"]
    res: dict = {"n_designs": B["n_designs"]}

    # P1 colorway, per family and pooled (train families vs held-out families)
    per_fam = {f: identification(Z[B["p1"][f]], lab(B["p1"][f]), Z[g], lab(g)) for f in FAMILIES}
    res["P1_per_family"] = per_fam
    for name, fams in (("all", FAMILIES), ("train_families", rc.TRAIN_FAMILIES),
                       ("heldout_families", rc.HELDOUT_FAMILIES)):
        q = [i for f in fams for i in B["p1"][f]]
        res[f"P1_{name}"] = identification(Z[q], lab(q), Z[g], lab(g))
    # P2 colour trap
    t = B["trap"]
    res["P2_trap"] = identification(Z[t], lab(t), Z[g], lab(g), donor_lab=[S[i]["donor_label"] for i in t])
    # P3 shared palette
    p3 = [identification(Z[q], lab(q), Z[gg], lab(gg)) for gg, q in B["shared"]]
    res["P3_shared_palette"] = {k: float(np.mean([r[k] for r in p3])) for k in p3[0]}
    # P4 real re-photos
    if B["rephoto"]:
        r = B["rephoto"]
        res["P4_rephoto"] = identification(Z[r], lab(r), Z[g], lab(g))

    # verification
    P = B["pairs"]
    cos = lambda pairs: (Z[[a for a, _ in pairs]] * Z[[b for _, b in pairs]]).sum(1).numpy()
    sg, sr, ss = cos(P["genuine"]), cos(P["impostor_random"]), cos(P["impostor_samepal"])
    res["verif_all"] = verification(np.r_[sg, sr, ss], np.r_[np.ones_like(sg), np.zeros_like(sr), np.zeros_like(ss)],
                                    threshold)
    res["verif_samepalette_only"] = verification(np.r_[sg, ss], np.r_[np.ones_like(sg), np.zeros_like(ss)], threshold)
    res["score_stats"] = {k: [float(v.mean()), float(v.std())] for k, v in
                          (("genuine", sg), ("impostor_random", sr), ("impostor_samepal", ss))}
    return res


def headline(res: dict) -> dict:
    """The handful of numbers worth putting in a table."""
    return {
        "P1 R@1": res["P1_all"]["R@1"], "P1 mAP": res["P1_all"]["mAP"],
        "P1 R@1 held-out fam": res["P1_heldout_families"]["R@1"],
        "P2 R@1": res["P2_trap"]["R@1"], "P2 Trap@1 (lower=better)": res["P2_trap"]["Trap@1"],
        "P3 R@1": res["P3_shared_palette"]["R@1"],
        **({"P4 R@1": res["P4_rephoto"]["R@1"]} if "P4_rephoto" in res else {}),
        "Verif AUC": res["verif_all"]["AUC"], "Verif EER": res["verif_all"]["EER"],
        "TAR@FAR=1%": res["verif_all"]["TAR@FAR=0.01"],
        "Same-palette AUC": res["verif_samepalette_only"]["AUC"],
        **({"Acc@val-thr": res["verif_all"]["acc@thr"]} if "acc@thr" in res["verif_all"] else {}),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--ckpt")
    ap.add_argument("--zero-shot", help="timm backbone name for a no-training baseline")
    ap.add_argument("--gray", action="store_true", help="feed grayscale (zero-shot ablation)")
    ap.add_argument("--split", default="test")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--tta", action="store_true")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", required=True, help="output json path")
    a = ap.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if a.ckpt:
        model, ck = load_checkpoint(a.ckpt)
        gray = ck.get("gray", False) or a.gray
        name = Path(a.ckpt).parent.name
    else:
        model, gray, name = zero_shot_model(a.zero_shot), a.gray, f"zeroshot_{a.zero_shot}" + ("_gray" if a.gray else "")
    model.to(device)
    kw = dict(gray=gray, seed=a.seed, workers=a.workers, tta=a.tta)
    val = run_benchmark(model, load_manifest(a.manifest, "val"), device, **kw)
    thr = val["verif_all"]["thr_at_EER"]
    test = run_benchmark(model, load_manifest(a.manifest, a.split), device, threshold=thr, **kw)
    out = {"model": name, "split": a.split, "val_threshold": thr, "headline": headline(test), "full": test}
    Path(a.out).parent.mkdir(parents=True, exist_ok=True)
    Path(a.out).write_text(json.dumps(out, indent=2))
    print(json.dumps(out["headline"], indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sareeid/train.py
"""Train the colour-invariant design embedder.

  python -m sareeid.train --manifest work/manifest.csv --out runs/atto --epochs 30

Each batch = P designs x K views. Every view gets an independent crop, a recolor (palette
remap / palette transfer from a shared bank / HSV / channel shuffle / gray) and capture
artefacts. Multi-positive SupCon pulls views of the same design together and pushes all
other designs apart, including designs that were given the very same palette.
Model selection uses the val-split benchmark (mean of P1, P2, P3 Recall@1).
"""
from __future__ import annotations

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import MultiViewTrainSet, load_manifest
from .evaluate import headline, run_benchmark
from .losses import supcon_loss
from .model import SareeEmbedder


def collate(batch):
    x = torch.cat([b[0] for b in batch])
    y = torch.tensor([b[1] for b in batch]).repeat_interleave(batch[0][0].shape[0])
    return x, y


def selection_score(res):
    return (res["P1_all"]["R@1"] + res["P2_trap"]["R@1"] + res["P3_shared_palette"]["R@1"]) / 3


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--backbone", default="convnext_atto.d2_in1k")
    ap.add_argument("--dim", type=int, default=256)
    ap.add_argument("--size", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--P", type=int, default=48, help="designs per batch")
    ap.add_argument("--views", type=int, default=4, help="views per design")
    ap.add_argument("--lr", type=float, default=3e-4, help="head lr; backbone gets lr*backbone_mult")
    ap.add_argument("--backbone-mult", type=float, default=0.3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--temperature", type=float, default=0.07)
    ap.add_argument("--warmup", type=int, default=1, help="warmup epochs")
    ap.add_argument("--eval-every", type=int, default=2)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--gray", action="store_true", help="ablation: train on grayscale input")
    ap.add_argument("--bank", type=int, default=24, help="shared palette bank size; 0 = ablation without sharing")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--max-steps", type=int, default=0, help="debug: cap steps per epoch")
    a = ap.parse_args()

    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(vars(a), indent=2))

    train_items, val_items = load_manifest(a.manifest, "train"), load_manifest(a.manifest, "val")
    ds = MultiViewTrainSet(train_items, size=a.size, views=a.views, bank_size=a.bank, seed=a.seed, gray=a.gray)
    P = min(a.P, len(ds))
    # persistent_workers=False so the per-epoch palette bank (set_epoch) reaches the workers
    dl = DataLoader(ds, batch_size=P, shuffle=True, drop_last=True, num_workers=a.workers,
                    collate_fn=collate, pin_memory=device.type == "cuda", persistent_workers=False)

    model = SareeEmbedder(a.backbone, a.dim, pretrained=True).to(device)
    bb = list(model.backbone.parameters())
    rest = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": bb, "lr": a.lr * a.backbone_mult}, {"params": rest, "lr": a.lr}],
                            weight_decay=a.wd)
    steps_per_epoch = min(len(dl), a.max_steps) if a.max_steps else len(dl)
    total, warm = a.epochs * steps_per_epoch, a.warmup * steps_per_epoch
    base = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

    hist, best, step = [], -1.0, 0
    for ep in range(a.epochs):
        ds.set_epoch(ep)
        model.train()
        t0, losses = time.time(), []
        for it, (x, y) in enumerate(dl):
            if a.max_steps and it >= a.max_steps:
                break
            f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g, b in zip(opt.param_groups, base):
                g["lr"] = b * f
            x, y = x.to(device, non_blocking=True), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                z = model(x)
            loss = supcon_loss(z, y, a.temperature)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            losses.append(loss.item()); step += 1
        rec = {"epoch": ep + 1, "loss": float(np.mean(losses)), "time_s": round(time.time() - t0, 1),
               "gem_p": model.pool.p.item()}
        if (ep + 1) % a.eval_every == 0 or ep + 1 == a.epochs:
            res = run_benchmark(model, val_items, device, gray=a.gray, workers=a.workers, size=a.size)
            rec["val"] = headline(res)
            rec["val_score"] = selection_score(res)
            if rec["val_score"] > best:
                best = rec["val_score"]
                torch.save({"state_dict": model.state_dict(), "backbone": a.backbone, "dim": a.dim,
                            "gray": a.gray, "size": a.size, "epoch": ep + 1,
                            "val_threshold": res["verif_all"]["thr_at_EER"], "val": rec["val"]}, out / "best.pt")
                rec["saved"] = True
        hist.append(rec)
        (out / "history.json").write_text(json.dumps(hist, indent=2))
        print(json.dumps(rec))
    print(f"[train] best val score {best:.4f} -> {out / 'best.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sareeid/infer.py
"""Inference CLI: build a gallery index, identify a query, verify a pair.

  python -m sareeid.infer index    --ckpt best.pt --images gallery_dir --out gallery.pt
  python -m sareeid.infer identify --ckpt best.pt --gallery gallery.pt --image q.jpg --topk 5
  python -m sareeid.infer verify   --ckpt best.pt --image a.jpg --image2 b.jpg

Pre-processing: decode -> RGB -> resize short side to `size` -> centre crop -> ImageNet norm.
Post-processing: L2-normalised embedding, (optional) h-flip TTA; gallery images are grouped by
design label (sub-folder name, else file stem) and a design scores the MAX over its colourways,
so adding more colourways of a design to the gallery can only help it.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import torch
import torch.nn.functional as F

from .data import IMG_EXT, center_crop, read_rgb, to_tensor
from .model import load_checkpoint


class SareeMatcher:
    def __init__(self, ckpt: str, device: str | None = None, tta: bool = True):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model, ck = load_checkpoint(ckpt)
        self.model.to(self.device)
        self.size, self.gray, self.tta = ck.get("size", 224), ck.get("gray", False), tta
        self.threshold = ck.get("val_threshold", 0.5)

    @torch.no_grad()
    def embed(self, paths: list[str], batch_size: int = 64) -> torch.Tensor:
        out = []
        for i in range(0, len(paths), batch_size):
            x = torch.stack([to_tensor(center_crop(read_rgb(p), self.size), self.gray)
                             for p in paths[i:i + batch_size]]).to(self.device)
            z = self.model(x)
            if self.tta:
                z = F.normalize(z + self.model(x.flip(-1)), dim=-1)
            out.append(z.cpu())
        return torch.cat(out)

    def build_index(self, folder: str) -> dict:
        root = Path(folder)
        paths = sorted(str(p) for p in root.rglob("*") if p.suffix.lower() in IMG_EXT)
        labels = [Path(p).parent.name if Path(p).parent != root else Path(p).stem for p in paths]
        return {"emb": self.embed(paths).half(), "paths": paths, "labels": labels}

    def identify(self, image: str, index: dict, topk: int = 5) -> list[tuple[str, float, str]]:
        q = self.embed([image])[0]
        sims = index["emb"].float() @ q
        best: dict[str, tuple[float, str]] = {}
        for s, lab, p in zip(sims.tolist(), index["labels"], index["paths"]):
            if lab not in best or s > best[lab][0]:
                best[lab] = (s, p)
        ranked = sorted(best.items(), key=lambda kv: -kv[1][0])[:topk]
        return [(lab, s, p) for lab, (s, p) in ranked]

    def verify(self, a: str, b: str) -> tuple[bool, float]:
        z = self.embed([a, b])
        s = float(z[0] @ z[1])
        return s >= self.threshold, s


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["index", "identify", "verify"])
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--images")
    ap.add_argument("--gallery")
    ap.add_argument("--image")
    ap.add_argument("--image2")
    ap.add_argument("--out", default="gallery.pt")
    ap.add_argument("--topk", type=int, default=5)
    a = ap.parse_args()
    m = SareeMatcher(a.ckpt)
    if a.cmd == "index":
        idx = m.build_index(a.images)
        torch.save(idx, a.out)
        print(f"indexed {len(idx['paths'])} images / {len(set(idx['labels']))} designs -> {a.out}")
    elif a.cmd == "identify":
        for rank, (lab, s, p) in enumerate(m.identify(a.image, torch.load(a.gallery), a.topk), 1):
            print(f"{rank}. {lab}  sim={s:.3f}  ({p})")
    else:
        same, s = m.verify(a.image, a.image2)
        print(f"{'SAME' if same else 'DIFFERENT'} design  sim={s:.3f}  threshold={m.threshold:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sareeid/efficiency.py
"""Params / FLOPs / latency / embedding size.

  python -m sareeid.efficiency --backbone convnext_atto.d2_in1k [--ckpt best.pt] --out eff.json
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import torch
from torch.utils.flop_counter import FlopCounterMode

from .model import SareeEmbedder, load_checkpoint


@torch.no_grad()
def latency_ms(model, x, n=50, warm=10):
    # fp16 on GPU via autocast (same as eval); GeM deliberately pools in fp32
    amp = torch.autocast(device_type="cuda", dtype=torch.float16, enabled=x.is_cuda)
    with amp:
        for _ in range(warm):
            model(x)
    if x.is_cuda:
        torch.cuda.synchronize()
    t = time.perf_counter()
    with amp:
        for _ in range(n):
            model(x)
    if x.is_cuda:
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000


def report(model: SareeEmbedder, size: int = 224) -> dict:
    model = model.eval()
    x = torch.randn(1, 3, size, size)
    with FlopCounterMode(display=False) as fc:
        model(x)
    out = {
        "backbone": model.backbone_name,
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
        # FlopCounterMode counts 1 MAC = 2 FLOPs; papers/timm usually quote GMACs as "GFLOPs"
        "GFLOPs_per_image": fc.get_total_flops() / 1e9,
        "GMACs_per_image": fc.get_total_flops() / 2e9,
        "embedding_dim": model.dim or model.backbone.num_features,
        "input": f"{size}x{size}",
        "cpu_threads": torch.get_num_threads(),
        "cpu_latency_ms_bs1": latency_ms(model, x),
    }
    out["embedding_bytes_fp16"] = out["embedding_dim"] * 2
    if torch.cuda.is_available():
        m = model.cuda()
        out["gpu"] = torch.cuda.get_device_name(0)
        out["gpu_latency_ms_bs1_fp16"] = latency_ms(m, x.cuda())
        xb = torch.randn(64, 3, size, size, device="cuda")
        out["gpu_throughput_img_s_bs64_fp16"] = 64 / (latency_ms(m, xb, n=20) / 1000)
        model.cpu()
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--backbone", default="convnext_atto.d2_in1k")
    ap.add_argument("--dim", type=int, default=256)
    ap.add_argument("--ckpt")
    ap.add_argument("--size", type=int, default=224)
    ap.add_argument("--out")
    a = ap.parse_args()
    m = load_checkpoint(a.ckpt)[0] if a.ckpt else SareeEmbedder(a.backbone, a.dim, pretrained=False)
    r = report(m, a.size)
    print(json.dumps(r, indent=2))
    if a.out:
        Path(a.out).write_text(json.dumps(r, indent=2))

In [ ]:
%%writefile sareeid/prepare.py
"""python -m sareeid.prepare --roots <dir> [<dir> ...] --out work"""
import argparse

from .data import prepare

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--roots", nargs="+", required=True, help="folders scanned recursively for images")
    ap.add_argument("--out", default="work")
    ap.add_argument("--short", type=int, default=320, help="cache resolution (short side)")
    ap.add_argument("--hash-thr", type=int, default=6, help="pHash Hamming radius for near-duplicate grouping")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    prepare(a.roots, a.out, short=a.short, hash_thr=a.hash_thr, seed=a.seed)

## 1. Data
Pipeline: every image folder under `/kaggle/input` → decode (corrupt and <96 px files dropped) → byte-level dedupe (MD5) → resize to short side 320 and cache → **near-duplicate grouping** with a perceptual hash matched across all 8 rotations/flips → **70/10/20 split by group**.

Grouping by near-duplicate means a re-shot, rotated or re-uploaded copy of a test saree can never appear in train.

In [ ]:
import glob
from pathlib import Path
# DeepLure corpus: the 'sarees_dataset' shortcut in My Drive (searched one level deep as a fallback)
cands = glob.glob('/content/drive/MyDrive/sarees_dataset') + glob.glob('/content/drive/MyDrive/*/sarees_dataset')
DEEPLURE = cands[0] if cands else None
print('DeepLure corpus:', DEEPLURE or 'NOT FOUND - add the Drive shortcut (see top cell)')
# Kaggle saree patterns via kagglehub
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    pass
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True); import kagglehub
try:
    KAGGLE_DIR = kagglehub.dataset_download('div456/indian-saree-patterns')
except Exception as e:
    KAGGLE_DIR = None; print('Kaggle dataset skipped:', e)
print('Kaggle dataset:', KAGGLE_DIR)
ROOTS = [r for r in (DEEPLURE, KAGGLE_DIR) if r]
assert ROOTS, 'No data found'

In [ ]:
import pandas as pd, importlib, sareeid.data
importlib.reload(sareeid.data)  # pick up the freshly written package file
print('dataset roots:', ROOTS)
sareeid.data.prepare(ROOTS, WORK)
man = pd.read_csv(f'{WORK}/manifest.csv')
display(man.groupby(['source','split']).size().unstack(fill_value=0))
gs = man.groupby('group').size()
print(f'{len(man)} images, {len(gs)} design groups, {(gs>1).sum()} groups with >1 near-duplicate photo')

## 2. Synthetic colorways
Each row is one design. Columns: original, then each recolor family. The last two (**curves**, **reinhard**) are *held out*: they are never used in training, only in evaluation.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sareeid import recolor as rc
from sareeid.data import read_rgb, center_crop
rng = np.random.default_rng(1)
paths = man[man.split=='test'].path.sample(4, random_state=0).tolist()
donor = center_crop(read_rgb(man.path.iloc[0]), 224); pal = rc.extract_palette(donor)
fams = ['original'] + list(rc.TRAIN_FAMILIES + rc.HELDOUT_FAMILIES)
fig, ax = plt.subplots(len(paths), len(fams), figsize=(2*len(fams), 2*len(paths)))
for r, p in enumerate(paths):
    im = center_crop(read_rgb(p), 224)
    for c, f in enumerate(fams):
        ax[r, c].imshow(im if f == 'original' else rc.apply_family(im, f, rng, target=pal, ref=donor))
        ax[r, c].axis('off'); ax[0, c].set_title(f, fontsize=9)
plt.tight_layout(); plt.show()

## 3. Evaluation protocol
Everything below runs on the held-out **test** split (designs never seen in training). The verification threshold is calibrated on **val** and applied unchanged to test.

| Protocol | Gallery | Query | What it measures |
|---|---|---|---|
| **P1 colorway** | 1 clean image per design | every design × 8 recolor families + crop/rotate/blur/JPEG | identification under recolor; separate score for the 2 held-out families |
| **P2 color trap** | same | design *d* painted in the exact palette of another design *d'* | R@1, and **Trap@1** = how often the palette donor is ranked first (color bias) |
| **P3 shared palette** | all designs in palette A | all designs in palette B | color carries zero information; only the motif can identify |
| **P4 re-photo** | same | real 2nd photos from near-duplicate groups | real capture variation (only when such groups exist) |
| **Verification** | pairs | genuine (cross-palette) vs impostor (random) and impostor (**same palette, different motif**) | AUC, EER, TAR@FAR=1%, accuracy at val threshold |

Every benchmark image comes from a seeded spec, so all models see byte-identical benchmarks.

## 4. Baselines (no training)
The ImageNet backbone used as-is, on RGB input and on grayscale input. Grayscale is the naive fix for color invariance; it fails when two different dyes have the same luminance, and when colorways invert light/dark.

In [ ]:
BACKBONE = 'convnext_atto.d2_in1k'
!python -m sareeid.evaluate --manifest {WORK}/manifest.csv --zero-shot {BACKBONE} --out {W}/results/zs_rgb.json --workers {NW}
!python -m sareeid.evaluate --manifest {WORK}/manifest.csv --zero-shot {BACKBONE} --gray --out {W}/results/zs_gray.json --workers {NW}

## 5. Train
PK batches of 48 designs × 4 views; SupCon with τ=0.07; AdamW (head 3e-4, backbone 0.3×), cosine schedule with 1 warmup epoch, AMP. The checkpoint is selected on the val benchmark (mean R@1 of P1, P2 and P3).

In [ ]:
EPOCHS = 30
!python -m sareeid.train --manifest {WORK}/manifest.csv --out {W}/runs/atto_rgb --backbone {BACKBONE} --epochs {EPOCHS} --P 48 --views 4 --eval-every 5 --workers {NW}

In [ ]:
import json
h = json.load(open(f'{W}/runs/atto_rgb/history.json'))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r['epoch'] for r in h], [r['loss'] for r in h]); ax[0].set_title('train SupCon loss'); ax[0].set_xlabel('epoch')
ev = [r for r in h if 'val' in r]
for k in ['P1 R@1', 'P2 R@1', 'P3 R@1', 'Same-palette AUC']:
    ax[1].plot([r['epoch'] for r in ev], [r['val'][k] for r in ev], marker='o', label=k)
ax[1].set_title('val benchmark'); ax[1].legend(); ax[1].set_xlabel('epoch'); plt.show()

In [ ]:
!python -m sareeid.evaluate --manifest {WORK}/manifest.csv --ckpt {W}/runs/atto_rgb/best.pt --out {W}/results/atto_rgb.json --workers {NW}

### Ablations (optional, set `RUN_ABLATIONS=True`)
- **gray**: the same training on grayscale input. Does the RGB model actually use chromatic edges?
- **no palette bank**: palette-transfer targets drawn independently per sample instead of from a shared bank (the `--bank 0` path).
- **bigger backbone**: ConvNeXt-Tiny (28M). How much accuracy does the lean model give up?

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    !python -m sareeid.train --manifest {WORK}/manifest.csv --out {W}/runs/atto_gray --backbone {BACKBONE} --epochs {EPOCHS} --gray --eval-every 5 --workers {NW}
    !python -m sareeid.evaluate --manifest {WORK}/manifest.csv --ckpt {W}/runs/atto_gray/best.pt --out {W}/results/atto_gray.json --workers {NW}
    !python -m sareeid.train --manifest {WORK}/manifest.csv --out {W}/runs/atto_nobank --backbone {BACKBONE} --epochs {EPOCHS} --bank 0 --eval-every 5 --workers {NW}
    !python -m sareeid.evaluate --manifest {WORK}/manifest.csv --ckpt {W}/runs/atto_nobank/best.pt --out {W}/results/atto_nobank.json --workers {NW}
    !python -m sareeid.train --manifest {WORK}/manifest.csv --out {W}/runs/tiny_rgb --backbone convnext_tiny.in12k_ft_in1k --epochs {EPOCHS} --P 32 --eval-every 5 --workers {NW}
    !python -m sareeid.evaluate --manifest {WORK}/manifest.csv --ckpt {W}/runs/tiny_rgb/best.pt --out {W}/results/tiny_rgb.json --workers {NW}

## 6. Results (test split)

In [ ]:
rows = {}
for f in sorted(glob.glob(f'{W}/results/*.json')):
    r = json.load(open(f)); rows[Path(f).stem] = r['headline']
res = pd.DataFrame(rows).T
display(res.style.format('{:.3f}').background_gradient(axis=0, cmap='Greens'))
print(res.to_markdown(floatfmt='.3f'))

In [ ]:
full = json.load(open(f'{W}/results/atto_rgb.json'))['full']
zs = json.load(open(f'{W}/results/zs_rgb.json'))['full']
fam = list(full['P1_per_family'])
x = np.arange(len(fam))
plt.figure(figsize=(10, 3.5))
plt.bar(x-0.2, [zs['P1_per_family'][f]['R@1'] for f in fam], 0.4, label='zero-shot RGB')
plt.bar(x+0.2, [full['P1_per_family'][f]['R@1'] for f in fam], 0.4, label='ours')
plt.xticks(x, [f + (' *' if f in rc.HELDOUT_FAMILIES else '') for f in fam], rotation=20)
plt.ylabel('R@1'); plt.title('P1 R@1 per recolor family (* = held out from training)'); plt.legend(); plt.show()
for name, r in [('zero-shot', zs), ('ours', full)]:
    print(name, 'cosine mean±sd', {k: f'{m:.2f}±{s:.2f}' for k, (m, s) in r['score_stats'].items()})

## 7. Qualitative check: color-trap queries
Each query is a test design painted in another design's palette. Green = the correct design, red = the palette donor (a color-biased model's answer).

In [ ]:
import torch
from sareeid.benchmark import build_specs, SpecDataset, embed_specs
from sareeid.data import load_manifest
from sareeid.model import load_checkpoint
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_items = load_manifest(f'{WORK}/manifest.csv', 'test')
B = build_specs(test_items, seed=0); S = B['specs']; ds = SpecDataset(S)
model, ck = load_checkpoint(f'{W}/runs/atto_rgb/best.pt'); model.to(dev)
idx = B['gallery'] + B['trap']
Z = embed_specs(model, [S[i] for i in idx], dev, workers=NW)
G, Q = Z[:len(B['gallery'])], Z[len(B['gallery']):]
sel = np.random.default_rng(0).choice(len(Q), 6, replace=False)
fig, ax = plt.subplots(len(sel), 6, figsize=(12, 2.1*len(sel)))
for r, qi in enumerate(sel):
    s = S[B['trap'][qi]]; top = (G @ Q[qi]).argsort(descending=True)[:5].tolist()
    ax[r, 0].imshow(ds.render(s)); ax[r, 0].set_title('query', fontsize=8)
    for c, g in enumerate(top, 1):
        ax[r, c].imshow(ds.render(S[B['gallery'][g]]))
        col = 'green' if g == s['label'] else 'red' if g == s['donor_label'] else 'gray'
        for sp in ax[r, c].spines.values(): sp.set_color(col); sp.set_linewidth(4)
        ax[r, c].set_title(f'#{c} {float(G[g] @ Q[qi]):.2f}', fontsize=8, color=col)
    for a in ax[r]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 8. Efficiency

In [ ]:
effs = {}
for run in sorted(glob.glob(f'{W}/runs/*/best.pt')):
    out = f'{W}/results/eff_{Path(run).parent.name}.json'
    !python -m sareeid.efficiency --ckpt {run} --out {out}
    effs[Path(run).parent.name] = json.load(open(out))
display(pd.DataFrame(effs).T)

## 9. Inference API
Index a gallery folder (sub-folder name = design id, so several colorways of one design can sit in one folder), then identify and verify.

In [ ]:
from sareeid.infer import SareeMatcher
import shutil, cv2
m = SareeMatcher(f'{W}/runs/atto_rgb/best.pt')
gal = Path(f'{W}/demo_gallery'); shutil.rmtree(gal, ignore_errors=True); gal.mkdir()
for it in test_items[:200]:
    (gal / f'design_{it.group}').mkdir(exist_ok=True); shutil.copy(it.path, gal / f'design_{it.group}')
index = m.build_index(str(gal))
q = rc.palette_remap(read_rgb(test_items[7].path), np.random.default_rng(3), mode='random')
cv2.imwrite(f'{W}/query.jpg', cv2.cvtColor(q, cv2.COLOR_RGB2BGR))
print('truth: design', test_items[7].group)
for lab, s, p in m.identify(f'{W}/query.jpg', index, topk=5): print(f'{lab:>14s}  {s:.3f}')
print('verify(query, its original):', m.verify(f'{W}/query.jpg', test_items[7].path))
print('verify(query, other design):', m.verify(f'{W}/query.jpg', test_items[8].path))

## 10. Clean up
The DeepLure corpus is proprietary. Once the exercise is done, delete the private Kaggle dataset and `/kaggle/working/work/cache`.